<a href="https://colab.research.google.com/github/Ridd1kulusC0d3r/OSINTIA/blob/main/notebooks/03_deepseek_r1_ach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 03 · Análise de Hipóteses Concorrentes (ACH) com DeepSeek-R1

**OSINTIA** · modelos de IA para OSINT, reconstruídos e auditáveis · [repositório](https://github.com/Ridd1kulusC0d3r/OSINTIA)

O **DeepSeek-R1-Distill-Qwen-7B** é um modelo de raciocínio que cabe no T4 em 4 bits. Aqui ele preenche a matriz de **ACH** (Heuer) sobre um caso fictício. A inovação: rodamos o raciocínio **várias vezes** e medimos a consistência de cada célula, a mesma lógica do teto do professor aplicada ao raciocínio.

| | |
|---|---|
| **Hardware** | GPU T4 gratuita (Ambiente de execução → Alterar tipo → T4 GPU) |
| **Tempo estimado** | 10 a 20 minutos |
| **Como rodar** | Menu *Ambiente de execução* → *Executar tudo* |

> ⚖️ Uso ético: só fontes abertas, finalidade legítima e respeito à LGPD. Os dados de demonstração são **fictícios**. Leia `docs/METODOLOGIA.md`.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
# Baixa o OSINTIA (núcleo em Python puro: calibração, auditoria, roteamento, custódia)
!git clone -q https://github.com/Ridd1kulusC0d3r/OSINTIA.git /content/OSINTIA 2>/dev/null || git -C /content/OSINTIA pull -q
import sys
sys.path.insert(0, "/content/OSINTIA")
from osintia import auditoria, calibracao, custodia, dados, esquemas, extracao, roteamento
print("OSINTIA pronto")

## 1. Carregar o modelo em 4 bits
Sem GPU grande? Troque por `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODELO = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
tok = AutoTokenizer.from_pretrained(MODELO)
llm = AutoModelForCausalLM.from_pretrained(
    MODELO,
    device_map = "auto",
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
)

## 2. O caso (fictício)
Campanha de perfis imitando o Banco Aurora. Quatro hipóteses, seis evidências.

In [ ]:
HIPOTESES = {
    "H1": "Golpistas oportunistas independentes",
    "H2": "Grupo organizado coordenando a campanha",
    "H3": "Teste autorizado de red team contratado pelo banco",
    "H4": "Perfis criados por erro interno do próprio banco",
}
EVIDENCIAS = {
    "E1": "12 perfis criados no mesmo intervalo de 48 horas",
    "E2": "Todos usam a mesma chave Pix para receber pagamentos",
    "E3": "As bios têm erros de português idênticos",
    "E4": "O banco nega ter contratado testes nesse período",
    "E5": "Vítimas relatam prejuízo real de dinheiro",
    "E6": "Três perfis compraram selo de verificado",
}

## 3. Pedir a matriz ACH em JSON
Recomendação dos autores do R1: sem *system prompt*, instruções na mensagem do usuário, temperatura ~0.6. O raciocínio vem entre `<think>` e `</think>`; guardamos, mas a decisão sai do JSON.

In [ ]:
import json, re

ESCALA = {"CC": 2, "C": 1, "N": 0, "I": -1, "II": -2}

def montar_prompt():
    h = "\n".join(f"{k}: {v}" for k, v in HIPOTESES.items())
    e = "\n".join(f"{k}: {v}" for k, v in EVIDENCIAS.items())
    return f"""Você é analista de inteligência e aplica Análise de Hipóteses Concorrentes (ACH).
Hipóteses:
{h}
Evidências:
{e}
Para cada evidência e hipótese, classifique: CC (muito consistente), C (consistente), N (neutra), I (inconsistente), II (muito inconsistente).
Responda ao final apenas com JSON no formato {{"E1": {{"H1": "C", "H2": "CC", ...}}, ...}}."""

def perguntar(prompt, temperatura=0.6, max_tokens=2048):
    msgs = [{"role": "user", "content": prompt}]
    entrada = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(llm.device)
    saida = llm.generate(entrada, max_new_tokens=max_tokens, do_sample=True, temperature=temperatura, top_p=0.95)
    return tok.decode(saida[0][entrada.shape[1]:], skip_special_tokens=True)

def separar(resposta):
    raciocinio, _, final = resposta.partition("</think>")
    if not final:
        final, raciocinio = raciocinio, ""
    blocos = re.findall(r"\{.*\}", final, flags=re.S)
    matriz = json.loads(blocos[-1]) if blocos else None
    return raciocinio.strip(), matriz

## 4. Rodar várias vezes (autoconsistência)

In [ ]:
RODADAS = 3
matrizes, raciocinios = [], []
for i in range(RODADAS):
    raciocinio, matriz = separar(perguntar(montar_prompt()))
    if matriz:
        matrizes.append(matriz); raciocinios.append(raciocinio)
    print(f"rodada {i + 1}: {'ok' if matriz else 'sem JSON válido'}")

## 5. Pontuar como Heuer
Na ACH a hipótese mais forte é a com **menos inconsistência**, não a com mais apoio. Somamos só os pesos negativos.

In [ ]:
from collections import Counter

def inconsistencia(matriz):
    return {h: sum(min(ESCALA.get(matriz.get(e, {}).get(h, "N"), 0), 0) for e in EVIDENCIAS) for h in HIPOTESES}

for i, m in enumerate(matrizes, 1):
    print(f"rodada {i}:", dict(sorted(inconsistencia(m).items(), key=lambda x: x[1], reverse=True)))

consistencia = {}
for e in EVIDENCIAS:
    for h in HIPOTESES:
        votos = Counter(m.get(e, {}).get(h, "N") for m in matrizes)
        rotulo, n = votos.most_common(1)[0]
        consistencia[(e, h)] = (rotulo, n / len(matrizes))

instaveis = [(k, v) for k, v in consistencia.items() if v[1] < 1]
print(f"\n{len(instaveis)} de {len(consistencia)} células mudaram entre rodadas:")
for (e, h), (rot, frac) in instaveis:
    print(f"  {e} x {h}: maioria {rot} em {frac:.0%}")

### Leitura
- Células instáveis são onde o modelo **não sabe**: leve essas para o analista.
- O raciocínio em `<think>` é útil para entender o modelo, mas **não é evidência**. Nunca cite como fonte.

In [ ]:
import os
os.makedirs("/content/saidas", exist_ok=True)
with open("/content/saidas/ach.json", "w", encoding="utf-8") as f:
    json.dump({"modelo": MODELO, "matrizes": matrizes, "raciocinios": raciocinios}, f, ensure_ascii=False, indent=2)
custodia.registrar("/content/saidas/manifesto.jsonl", "ach_deepseek", "/content/saidas/ach.json", {"rodadas": RODADAS})

---
### Próximos passos
- `docs/ENGENHARIA-REVERSA.md` explica como cada peça foi reconstruída.
- `docs/AUDITORIA-DE-ROTULOS.md` mostra por que acurácia alta pode esconder erro herdado.
- Abra uma *issue* no repositório se algo quebrar no Colab: as bibliotecas de IA mudam rápido.

*OSINTIA · Deivison Lourenço · MIT*